# Treatment delay and mortality in low-severity ED visits

## TL;DR

The doubly robust AIPW estimate is **+1.43 percentage points** for delay ≥160 minutes versus <160 minutes. The simple bootstrap 95% CI is **−0.07 to +2.77 points**, so the signal is not conclusive.

## Context & Methods

- Cohort: severity levels 1–3
- Confounders: severity and age
- Estimator: five-fold cross-fitted propensity-score AIPW
- Uncertainty: 200-replicate percentile bootstrap

### Key assumptions

Consistency, positivity, correct time ordering, and no important unmeasured confounding.

In [1]:
from pathlib import Path
import os
import sys
import pandas as pd

REPO = Path.cwd().resolve()
if not (REPO / "analysis.py").exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from analysis import load_data, run_analysis

DATA = Path(os.environ.get("MORTALITY_XLSX", REPO / "data" / "raw" / "mortality.xlsx"))

## Data

In [2]:
data, cohort = load_data(DATA)
pd.DataFrame({
    "All visits": [len(data), int(data["Mortality_Flag"].sum())],
    "Severity 1–3": [len(cohort), int(cohort["Mortality_Flag"].sum())],
}, index=["Visits", "Deaths"])

,All visits,Severity 1–3
Visits,9994,5216
Deaths,332,102


## Results

The pooled comparison reverses after defining the low-severity cohort.

![Crude mortality reversal](../figures/01_crude_mortality_reversal.png)

Severity drives both treatment timing and mortality.

![Severity mechanism](../figures/02_severity_mechanism.png)

In [3]:
result = run_analysis(DATA, REPO, threshold=160, bootstraps=200)
pd.Series({
    "Adjusted risk — high delay (%)": result["risk_high"] * 100,
    "Adjusted risk — lower delay (%)": result["risk_low"] * 100,
    "AIPW risk difference (pp)": result["risk_difference"] * 100,
    "95% CI lower (pp)": result["ci_low"] * 100,
    "95% CI upper (pp)": result["ci_high"] * 100,
}).round(2)

Adjusted risk — high delay (%)     3.21
Adjusted risk — lower delay (%)    1.79
AIPW risk difference (pp)          1.43
95% CI lower (pp)                 -0.07
95% CI upper (pp)                  2.77
dtype: float64

![Doubly robust effect estimate](../figures/03_effect_estimate.png)

## Takeaways

- The overall average is confounded by severity.
- The adjusted estimate suggests possible harm from long delay.
- The confidence interval includes zero; this is a signal to investigate, not a clinical threshold.